# Python Tutorial — Chapter 15: Modules and Packages

**Topics Covered:**
- What is a module and what is a package?
- Creating your own module
- The `if __name__ == "__main__"` guard
- Creating a package with multiple modules
- Module in a subdirectory — importing from a parent directory
- Module in a subdirectory — importing from a parallel (sibling) directory
- Package in a subdirectory — same patterns
- Exploring modules with `dir()` and `help()`
- How Python finds modules: `sys.path`

---

## 1. Modules vs Packages

A **module** is simply a `.py` file. When you write `converter.py` and save it, that file becomes a module you can import anywhere.

A **package** is a folder that contains several modules along with a special file called `__init__.py`. That file tells Python to treat the folder as an importable unit.

Think of it like a bookshelf: a module is one book, and a package is a labelled section of the shelf that groups related books together.

In [ ]:
# Visualising the difference
structure = """
MODULE (single .py file)
------------------------
converter.py          <-- import converter

PACKAGE (folder of modules)
----------------------------
mypackage/
    __init__.py       <-- marks the folder as a package
    shapes.py         <-- import mypackage.shapes
    colors.py         <-- import mypackage.colors
"""
print(structure)

---
## 2. Creating Your Own Module

Any `.py` file you write is automatically a module. The cells below:
1. Create a file called `converter.py` in the same folder as this notebook using `Path.write_text()`
2. Import it just like any other module

This is exactly how you would create a module in a real project — you just write a `.py` file.

In [ ]:
# Create converter.py in the same folder as this notebook
from pathlib import Path

Path('converter.py').write_text('''
"""converter.py — a simple unit conversion module."""


def km_to_miles(km):
    """Convert kilometres to miles."""
    return km * 0.621371


def celsius_to_fahrenheit(c):
    """Convert Celsius to Fahrenheit."""
    return c * 9 / 5 + 32


def kg_to_pounds(kg):
    """Convert kilograms to pounds."""
    return kg * 2.20462
'''.strip())

print('converter.py created successfully.')
print('Contents:')
print(Path('converter.py').read_text())

In [ ]:
# Import and use it — just like any standard library module
import converter

print(converter.km_to_miles(10))            # 6.21 miles
print(converter.celsius_to_fahrenheit(100)) # 212.0 F
print(converter.kg_to_pounds(70))           # 154.32 lbs

In [ ]:
# Import specific names with 'from ... import'
from converter import km_to_miles, kg_to_pounds

print(f'A marathon is {km_to_miles(42.195):.2f} miles.')
print(f'5 kg is {kg_to_pounds(5):.2f} lbs.')

In [ ]:
# Give the module an alias so it is shorter to type
import converter as cv

for t in [0, 20, 37, 100]:
    print(f'{t}°C = {cv.celsius_to_fahrenheit(t):.1f}°F')

---
## 3. The `if __name__ == "__main__"` Guard

Every Python file has a built-in variable called `__name__`.
- When you **run** a file directly (`python converter.py`), Python sets `__name__` to `'__main__'`.
- When that same file is **imported**, `__name__` is set to the module name (e.g. `'converter'`).

The guard `if __name__ == "__main__":` lets you put test or script code inside a module that only runs when the file is executed directly — it is skipped on import.

In [ ]:
# Rewrite converter.py — add the __main__ guard
from pathlib import Path

Path('converter.py').write_text('''
"""converter.py — unit conversion module with __main__ guard."""


def km_to_miles(km):
    """Convert kilometres to miles."""
    return km * 0.621371


def celsius_to_fahrenheit(c):
    """Convert Celsius to Fahrenheit."""
    return c * 9 / 5 + 32


def kg_to_pounds(kg):
    """Convert kilograms to pounds."""
    return kg * 2.20462


if __name__ == "__main__":
    # Runs ONLY when the file is executed directly — skipped on import
    print("Running converter.py as a script:")
    print(f"  10 km  = {km_to_miles(10):.3f} miles")
    print(f"  100 C  = {celsius_to_fahrenheit(100):.1f} F")
    print(f"  70 kg  = {kg_to_pounds(70):.2f} lbs")
'''.strip())

print('converter.py updated with __main__ guard.')

In [ ]:
# Importing the updated module — the if __name__ block is NOT executed
import importlib
import converter
importlib.reload(converter)  # reload so Python picks up the updated file

# No script output here — the guard worked
print('Import complete. Guard blocked the script output.')
print('km_to_miles(5) =', converter.km_to_miles(5))

In [ ]:
# Run the file directly as a script — now the guard block DOES execute
import subprocess, sys
result = subprocess.run([sys.executable, 'converter.py'], capture_output=True, text=True)
print(result.stdout)

---
## 4. Creating a Package

A package is a **folder** that contains:
- An `__init__.py` file (can be empty — its presence makes the folder a package)
- One or more `.py` module files

**Steps to create a package:**
1. Create a new folder (e.g. `mypackage/`)
2. Add `__init__.py` inside it
3. Add your module `.py` files inside the folder

```
mypackage/
    __init__.py    <-- required: marks the folder as a package
    shapes.py      <-- module 1
    colors.py      <-- module 2
```

In [ ]:
# Step 1: Create the package folder and __init__.py
from pathlib import Path

pkg = Path('mypackage')
pkg.mkdir(exist_ok=True)

(pkg / '__init__.py').write_text('# mypackage initialiser\n')

print('Created mypackage/__init__.py')

In [ ]:
# Step 2: Create mypackage/shapes.py
from pathlib import Path

Path('mypackage/shapes.py').write_text('''
"""shapes.py — basic geometry calculations."""
import math


def circle_area(radius):
    """Return the area of a circle."""
    return math.pi * radius ** 2


def rectangle_area(width, height):
    """Return the area of a rectangle."""
    return width * height


def triangle_area(base, height):
    """Return the area of a triangle."""
    return 0.5 * base * height
'''.strip())

print('Created mypackage/shapes.py')

In [ ]:
# Step 3: Create mypackage/colors.py
from pathlib import Path

Path('mypackage/colors.py').write_text('''
"""colors.py — simple colour helpers."""

NAMED_COLORS = {
    'red':   '#FF0000',
    'green': '#00FF00',
    'blue':  '#0000FF',
    'white': '#FFFFFF',
    'black': '#000000',
}


def hex_to_rgb(hex_color):
    """Convert a hex colour string to an (R, G, B) tuple."""
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))


def color_name_to_hex(name):
    """Return the hex code for a named colour, or None if unknown."""
    return NAMED_COLORS.get(name.lower())
'''.strip())

print('Created mypackage/colors.py')

# Show the final structure
print('\nPackage structure:')
for p in sorted(Path('mypackage').rglob('*')):
    print(' ', p)

In [ ]:
# Import from the package
from mypackage import shapes
from mypackage.colors import hex_to_rgb, color_name_to_hex

print('Circle area (r=5):', round(shapes.circle_area(5), 2))
print('Rectangle area (4x6):', shapes.rectangle_area(4, 6))
print('Triangle area (base=8, h=3):', shapes.triangle_area(8, 3))

print('\nred ->', color_name_to_hex('red'))
print('#1A2B3C as RGB ->', hex_to_rgb('#1A2B3C'))

---
## 5. Module in a Subdirectory

So far the module lived in the **same folder** as the notebook. In real projects you often place modules inside a subdirectory.

```
PythonBasics/          <-- current working directory (parent)
    mymodules/         <-- subdirectory
        converter.py   <-- module lives here
    notebook.ipynb     <-- we import from here
```

Python does **not** automatically search subdirectories. You must add the subdirectory to `sys.path`.

In [ ]:
# Step 1: Create the subdirectory and write the module inside it
from pathlib import Path

Path('mymodules').mkdir(exist_ok=True)

Path('mymodules/converter.py').write_text('''
"""converter.py — inside mymodules/ subdirectory."""


def km_to_miles(km):
    """Convert kilometres to miles."""
    return km * 0.621371


def celsius_to_fahrenheit(c):
    """Convert Celsius to Fahrenheit."""
    return c * 9 / 5 + 32


def kg_to_pounds(kg):
    """Convert kilograms to pounds."""
    return kg * 2.20462


if __name__ == "__main__":
    print("Running converter.py directly")
    print(f"  10 km = {km_to_miles(10):.3f} miles")
'''.strip())

print('Created: mymodules/converter.py')
print('Contents:', list(Path('mymodules').iterdir()))

In [ ]:
# Step 2: Import from the parent directory (this notebook)
# sys.path must contain the PARENT of the module — so add mymodules/ itself
import sys
from pathlib import Path

subdir = str(Path('mymodules').resolve())
if subdir not in sys.path:
    sys.path.insert(0, subdir)

import converter  # Python now finds mymodules/converter.py

print('Imported from subdirectory mymodules/')
print(f'  10 km  = {converter.km_to_miles(10):.3f} miles')
print(f'  100 C  = {converter.celsius_to_fahrenheit(100):.1f} F')
print(f'  70 kg  = {converter.kg_to_pounds(70):.2f} lbs')

### 5.1 Importing from a Parallel (Sibling) Directory

Sometimes a script needs to import a module that lives in a sibling folder:

```
PythonBasics/
    mymodules/
        converter.py   <-- module
    scripts/
        main.py        <-- wants to import converter from sibling mymodules/
```

Inside `scripts/main.py` you navigate using `__file__`:

```python
import sys, os

# __file__ = .../scripts/main.py
# os.path.dirname(__file__) = .../scripts/
# '..' goes up one level to PythonBasics/
# then into mymodules/ — the sibling directory
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'mymodules'))

import converter
```

The cell below creates `scripts/main.py` and runs it to demonstrate.

In [ ]:
# Create scripts/main.py and run it as a separate process
from pathlib import Path
import subprocess, sys

Path('scripts').mkdir(exist_ok=True)

Path('scripts/main.py').write_text('''
import sys
import os

# Navigate: scripts/ -> up one level (..) -> mymodules/
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'mymodules'))

import converter

print("Imported from sibling mymodules/ directory!")
print(f"  5 km  = {converter.km_to_miles(5):.3f} miles")
print(f"  37 C  = {converter.celsius_to_fahrenheit(37):.1f} F")
print(f"  60 kg = {converter.kg_to_pounds(60):.2f} lbs")
'''.strip())

result = subprocess.run([sys.executable, 'scripts/main.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('ERRORS:', result.stderr)

---
## 6. Package in a Subdirectory

The same `sys.path` pattern applies to packages. Place the package folder inside a subdirectory, add that subdirectory to `sys.path`, and import the package normally.

```
PythonBasics/
    mylibs/                    <-- subdirectory holding packages
        unitconverter/         <-- the package
            __init__.py
            length.py
            temperature.py
    notebook.ipynb             <-- we import from here
```

> **Key rule:** `sys.path` must point to the **parent** of what you want to import.
> - To import `converter` → add the folder that contains `converter.py`
> - To import `unitconverter` package → add the folder that contains the `unitconverter/` folder

In [ ]:
# Step 1: Create mylibs/unitconverter/ package structure
from pathlib import Path

pkg = Path('mylibs/unitconverter')
pkg.mkdir(parents=True, exist_ok=True)

(pkg / '__init__.py').write_text('''
"""unitconverter — unit conversion package inside mylibs/ subdirectory."""
__version__ = '1.0.0'
__author__  = 'Python Learner'
'''.strip())

(pkg / 'length.py').write_text('''
"""length.py — distance conversions."""


def km_to_miles(km):
    """Convert kilometres to miles."""
    return km * 0.621371


def miles_to_km(miles):
    """Convert miles to kilometres."""
    return miles / 0.621371


def meters_to_feet(m):
    """Convert metres to feet."""
    return m * 3.28084


def cm_to_inches(cm):
    """Convert centimetres to inches."""
    return cm * 0.393701
'''.strip())

(pkg / 'temperature.py').write_text('''
"""temperature.py — temperature conversions."""


def celsius_to_fahrenheit(c):
    """Convert Celsius to Fahrenheit."""
    return c * 9 / 5 + 32


def fahrenheit_to_celsius(f):
    """Convert Fahrenheit to Celsius."""
    return (f - 32) * 5 / 9


def celsius_to_kelvin(c):
    """Convert Celsius to Kelvin."""
    return c + 273.15


def kelvin_to_celsius(k):
    """Convert Kelvin to Celsius."""
    return k - 273.15
'''.strip())

print('Package structure created:')
for p in sorted(Path('mylibs').rglob('*')):
    print(' ', p)

In [ ]:
# Step 2: Import from the parent directory (this notebook)
# Add mylibs/ to sys.path — it is the PARENT of the unitconverter package folder
import sys
from pathlib import Path

mylibs = str(Path('mylibs').resolve())
if mylibs not in sys.path:
    sys.path.insert(0, mylibs)

from unitconverter import length, temperature

print('--- Package from subdirectory mylibs/ ---')
print(f'  Marathon 42.195 km  = {length.km_to_miles(42.195):.2f} miles')
print(f'  100 metres          = {length.meters_to_feet(100):.2f} feet')
print(f'  Body temp 37 C      = {temperature.celsius_to_fahrenheit(37):.1f} F')
print(f'  Boiling point 212 F = {temperature.fahrenheit_to_celsius(212):.1f} C')

### 6.1 Importing a Package from a Parallel Directory

```
PythonBasics/
    mylibs/
        unitconverter/     <-- package
    scripts/
        run.py             <-- wants to import unitconverter from sibling mylibs/
```

Inside `scripts/run.py`:

```python
import sys, os

# Go up one level from scripts/ to PythonBasics/, then into mylibs/
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'mylibs'))

from unitconverter import length
print(length.km_to_miles(10))
```

In [ ]:
# Create scripts/run.py and execute it to prove parallel-directory package import
from pathlib import Path
import subprocess, sys

Path('scripts').mkdir(exist_ok=True)

Path('scripts/run.py').write_text('''
import sys
import os

# Navigate from scripts/ -> up one level -> mylibs/ (the sibling directory)
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'mylibs'))

from unitconverter import length, temperature

print("Imported unitconverter package from sibling mylibs/ directory!")
print(f"  Marathon 42.195 km = {length.km_to_miles(42.195):.2f} miles")
print(f"  Body temp 37 C     = {temperature.celsius_to_fahrenheit(37):.1f} F")
print(f"  Absolute zero 0 K  = {temperature.kelvin_to_celsius(0):.2f} C")
'''.strip())

result = subprocess.run([sys.executable, 'scripts/run.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('ERRORS:', result.stderr)

### Summary: Import Locations at a Glance

| Where the module/package lives | What to add to `sys.path` |
|---|---|
| Same folder as the script | Nothing — Python searches here by default |
| Subdirectory `mymodules/` | `sys.path.insert(0, 'mymodules')` |
| Package in subdirectory `mylibs/unitconverter/` | `sys.path.insert(0, 'mylibs')` |
| Sibling directory (from a script) | `sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'sibling'))` |

> **Rule of thumb:** `sys.path` must contain the **parent folder** of the module or package — not the module/package folder itself.

---
## 7. Exploring Modules with `dir()` and `help()`

`dir(module)` lists every name defined in a module. `help(module)` renders its docstrings as readable documentation. Both work on your own modules and on standard library modules.

In [ ]:
import converter

# All names in the module
print('All names:', dir(converter))

# Filter out dunder names
public = [n for n in dir(converter) if not n.startswith('_')]
print('Public names:', public)

In [ ]:
# Read a function's docstring
help(converter.km_to_miles)

In [ ]:
# Works on standard library modules too
import math
math_public = [n for n in dir(math) if not n.startswith('_')]
print(f'math has {len(math_public)} public names:')
print(math_public)

---
## 8. How Python Finds Modules: `sys.path`

When you write `import something`, Python searches each directory in `sys.path` in order until it finds a match:
1. The current script's directory (or notebook's working directory)
2. Any paths in the `PYTHONPATH` environment variable
3. The standard library directories
4. Installed third-party packages (site-packages)

If Python cannot find the module in any of these, you get `ModuleNotFoundError`.

In [ ]:
import sys

print('Python searches these paths in order:')
for i, path in enumerate(sys.path):
    print(f'  [{i}] {path}')

In [ ]:
# Add a custom directory at runtime
import sys
from pathlib import Path

custom = str(Path('mymodules').resolve())
if custom not in sys.path:
    sys.path.insert(0, custom)
    print(f'Added to sys.path: {custom}')
else:
    print('Already in sys.path')

print('First entry:', sys.path[0])

---
## 9. Exercises

Try the following on your own.

In [ ]:
# Exercise 1:
# Create a module called greetings.py in a subdirectory called utils/.
# The module should have two functions:
#   say_hello(name)   -> returns 'Hello, <name>!'
#   say_goodbye(name) -> returns 'Goodbye, <name>! See you soon.'
# Add an if __name__ == '__main__' block that calls both functions.
# Then import the module from this notebook (parent directory) and call both functions.


In [ ]:
# Exercise 2:
# Create a package called mathtools in a subdirectory called libs/.
# The package should have one module stats.py with three functions:
#   mean(numbers)    -> returns the average of a list
#   minimum(numbers) -> returns the smallest value (without using min())
#   maximum(numbers) -> returns the largest value (without using max())
# Import the package from this notebook and test all three functions
# with the list [4, 7, 2, 9, 1, 5].


In [ ]:
# Exercise 3:
# Create a script called scripts/calc.py that imports the mathtools package
# from the sibling libs/ directory (using __file__ and sys.path).
# The script should print the mean, minimum, and maximum of [10, 3, 8, 1, 6].
# Run it from this notebook using subprocess and print the output.


---
## Key Takeaways

- A **module** is any `.py` file; a **package** is a folder of modules that contains `__init__.py`.
- Import styles: `import name`, `from name import func`, `import name as alias`.
- The `if __name__ == "__main__"` guard prevents script code from running when a module is imported.
- `__init__.py` makes a folder a package and can set metadata like `__version__`.
- `sys.path` is the ordered list of directories Python searches on every import — the current directory is always first.
- To import from a **subdirectory**: add that subdirectory to `sys.path`.
- To import from a **sibling directory**: use `__file__` + `..` to build the path, then add it to `sys.path`.
- `sys.path` must point to the **parent** of what you want to import, not the module/package itself.